Crea volumen, carpetas y tablas Bronze/Silver para el pronostico ECMWF (cf via TIGGE/cdsapi, pf via TIGGE/cdsapi - ensemble completo de 50 miembros perturbados). fc (Open Data) fue descartado: cfgrib/eccodes >=2.39 crashea en el compute serverless de este workspace (ver Decision 013 en docs/decisions.md), y el workspace no permite compute clasico. Bronze guarda todo el bounding box descargado; Silver aplica el recorte al poligono real de las 3 sub-cuencas.

In [ ]:
spark.sql('CREATE CATALOG IF NOT EXISTS weather')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.raw')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.silver')
spark.sql('CREATE VOLUME IF NOT EXISTS weather.raw.ecmwf_volume')

dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/cf_tigge/raw')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/cf_tigge/json')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/pf_tigge/raw')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/pf_tigge/json')


## Tablas Bronze

Espejo fiel de lo descargado (todo el bounding box, sin recortar al poligono). `tp_mm` ya viene normalizado a milimetros en el aplanado (cf/pf: kg/m2, ya equivalente a mm, sin conversion).

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ecmwf_forecast_cf (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ecmwf_forecast_pf (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')


## Tablas Silver

Solo los puntos de grilla dentro del buffer (~0.15 grados) del poligono union de las 3 sub-cuencas (`SIG/subcuencas_modelo.geojson`).

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_cf_basin (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_pf_basin (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

# Agregado por sub-cuenca (Decision 048): una fila por (run_date, run_time, step_hours, miembro,
# sub-cuenca), promediando los puntos de grilla de la sub-cuenca. Conserva los 50 miembros de pf
# -- promediarlos aca borraria la dispersion del ensemble, que es la unica medida de
# incertidumbre del pronostico; ese colapso es una decision de Gold.
spark.sql("""
CREATE TABLE IF NOT EXISTS weather.silver.punto_subcuenca (
  latitude DOUBLE,
  longitude DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING
) USING DELTA
""")

for _modelo in ('cf', 'pf'):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_{_modelo}_subcuenca (
      run_date DATE,
      run_time STRING,
      step_hours INT,
      valid_date DATE,
      valid_datetime TIMESTAMP,
      number INT,
      subcuenca_id INT,
      subcuenca_nombre STRING,
      tp_mm_medio DOUBLE,
      n_puntos BIGINT,
      fuente STRING,
      source_table STRING,
      processed_at TIMESTAMP,
      updated_at TIMESTAMP
    ) USING DELTA
    """)

In [ ]:
# Decision 053: relleno de `cf` para 2000-01->2006-09 (antes de que exista TIGGE) calibrando
# GEFS Reforecast v12 `c00` contra el sesgo real medido en el solapamiento 2006-10->2019-12.
# gefs_cf_bias es la tabla chica de auditoria (un valor de sesgo por sub-cuenca x lead_day);
# gefs_cf_fill_grid tiene el mismo shape que weather.bronze.ecmwf_forecast_cf (grilla completa,
# no un promedio) para que ETL_Silver_ECMWF_Subcuenca la trate igual que un dia real de cf y la
# agregacion a sub-cuenca (y su eventual cambio de metrica) no tenga que saber que existe.
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.gefs_cf_bias (
  subcuenca_nombre STRING,
  lead_day INT,
  bias_mm DOUBLE,
  n_dias_solapamiento BIGINT,
  metodo STRING,
  calibrado BOOLEAN,
  computed_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.gefs_cf_fill_grid (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  fuente STRING,
  bias_mm_aplicado DOUBLE
) USING DELTA
''')

# weather.silver.ecmwf_forecast_{cf,pf}_subcuenca ya existen en produccion sin esta columna --
# el CREATE TABLE IF NOT EXISTS de mas arriba no la agrega a una tabla que ya existe, hace
# falta ALTER. Las dos, no solo cf: ETL_Silver_ECMWF_Subcuenca siempre escribe 'fuente' ahora
# (para pf vale siempre 'tigge'), asi que si a pf le faltara la columna el MERGE del job
# diario rompe. Guardado con un chequeo de columnas existentes para que una corrida repetida
# no falle.
for _modelo_alter in ('cf', 'pf'):
    _tabla = f'weather.silver.ecmwf_forecast_{_modelo_alter}_subcuenca'
    _columnas = {f.name for f in spark.table(_tabla).schema.fields}
    if 'fuente' not in _columnas:
        spark.sql(f'ALTER TABLE {_tabla} ADD COLUMNS (fuente STRING)')
        print(f'{_tabla}: columna fuente agregada')
    else:
        print(f'{_tabla}: columna fuente ya existia')

In [ ]:
for table_name in [
    'weather.bronze.ecmwf_forecast_cf', 'weather.bronze.ecmwf_forecast_pf',
    'weather.silver.ecmwf_forecast_cf_basin', 'weather.silver.ecmwf_forecast_pf_basin',
    'weather.silver.punto_subcuenca',
    'weather.silver.ecmwf_forecast_cf_subcuenca', 'weather.silver.ecmwf_forecast_pf_subcuenca',
    'weather.silver.gefs_cf_bias', 'weather.silver.gefs_cf_fill_grid',
]:
    print(f'DESCRIBE {table_name}')
    spark.sql(f'DESCRIBE {table_name}').show(truncate=False)